ENTRY 1. Run all cells for each RUN, restarting kernel between methods. Vanilla is copied from your completed run. Training never constructs CIFAR-100.

In [1]:
RUN='proser' # {vanilla, gcsc, proser}
REUSE_PRESERVED_VANILLA=True

In [2]:
from pathlib import Path
import json, hashlib, random, math, shutil
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,Subset
from torchvision import datasets,models,transforms
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'configs/vanilla.yaml').exists()),None)
if ROOT is None:
    candidate=Path.cwd()/'Task 4/task4'
    if not candidate.exists():candidate=Path.cwd()/'task4'
    if not candidate.exists():raise RuntimeError('Open Jupyter in Task 4/task4.')
    ROOT=candidate.resolve()
DATA_ROOT=ROOT.parent.parent/'data'
DOWNLOAD=True # Downloads missing datasets only when YOU run a notebook.
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def read_json(p):return json.loads(Path(p).read_text(encoding='utf-8'))
def write_json(p,value):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(value,indent=2,allow_nan=False),encoding='utf-8')
def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def use(relative):
    path=ROOT/relative
    notebook=read_json(path)
    for i,cell in enumerate(notebook['cells']):
        if cell['cell_type']=='code':exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),globals())
def seed_all():
    random.seed(6304);np.random.seed(6304);torch.manual_seed(6304);torch.cuda.manual_seed_all(6304)
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
for p in ['data/make_splits.ipynb','data/cifar10.ipynb','models/resnet_cifar.ipynb','methods/vanilla.ipynb','methods/gcsc.ipynb','methods/manifold_mixup.ipynb','methods/proser.ipynb']:use(p)
print('Root:',ROOT,'Device:',DEVICE)


cfg=read_json(ROOT/'configs'/f'{RUN}.yaml')
out=ROOT/'results'/RUN;out.mkdir(parents=True,exist_ok=True)
if (ROOT/'results/protocol_lock.json').exists():raise RuntimeError('Protocol is locked. Use a new experiment folder to change models.')
seed_all()
train,clean,test,split=known_data(RUN)
train_loader=loader(train,split['train'],shuffle=True)
val_loader=loader(clean,split['validation'])
model=make_model().to(DEVICE)
if RUN=='proser':
    vanilla=ROOT/'results/vanilla/best.pt'
    if not (vanilla.parent/'complete.json').exists():raise RuntimeError('Complete Vanilla first.')
    state=torch.load(vanilla,map_location=DEVICE,weights_only=False)
    model.load_state_dict(state['model_state'])
    model.dummy=nn.Linear(512,5).to(DEVICE)
    cfg['vanilla_checkpoint_sha256']=sha(vanilla)
@torch.no_grad()
def validation_accuracy():
    model.eval();correct=count=0
    for x,y in val_loader:
        _,z,_=forward_outputs(model,x.to(DEVICE))
        correct+=(z.argmax(1).cpu()==y).sum().item();count+=len(y)
    return correct/count

def train_or_reuse():
    if (out/'complete.json').exists():
        if RUN=='vanilla' and not REUSE_PRESERVED_VANILLA:raise RuntimeError('Preserved baseline exists; use a separate experiment to retrain.')
        checkpoint=torch.load(out/'best.pt',map_location=DEVICE,weights_only=False)
        saved=checkpoint['config']
        if RUN=='vanilla':
            inventory=read_json(ROOT/'results/preserved_vanilla.json')
            if sha(out/'best.pt')!=inventory['best.pt']['sha256']:raise ValueError('Preserved checkpoint hash differs.')
        if RUN=='proser' and saved.get('vanilla_checkpoint_sha256')!=cfg['vanilla_checkpoint_sha256']:raise ValueError('PROSER initialization checkpoint differs.')
        for key in ['seed','epochs','batch_size','lr','momentum','weight_decay','normalization_mean','normalization_std']:
            if saved[key]!=cfg[key]:raise ValueError(f'Completed run has different {key}.')
        if checkpoint['split']!=split:raise ValueError('Checkpoint split mismatch.')
        if read_json(out/'complete.json')['epochs_completed']!=cfg['epochs']:raise ValueError('Incomplete run.')
        model.load_state_dict(checkpoint['model_state'])
        print('Reusing completed',RUN,'best validation accuracy:',checkpoint['validation_accuracy'])
        return
    if (out/'history.json').exists() or (out/'best.pt').exists():raise RuntimeError('Partial run exists. Preserve it elsewhere before restarting this run from epoch 1.')
    optimizer=torch.optim.SGD(model.parameters(),lr=cfg['lr'],momentum=cfg['momentum'],weight_decay=cfg['weight_decay'])
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=cfg['epochs'])
    write_json(out/'config.json',cfg);write_json(out/'split.json',split)
    history=[];best=-1.
    for epoch in range(cfg['epochs']):
        model.train();total=0.;n=0;correct=count=0;components={};batches=0
        for x,y in train_loader:
            x,y=x.to(DEVICE),y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss,detail=(proser_loss if RUN=='proser' else gcsc_loss if RUN=='gcsc' else ordinary_loss)(model,x,y)
            loss.backward();optimizer.step()
            total+=loss.item()*len(y);n+=len(y);batches+=1
            correct+=detail.pop('correct');count+=detail.pop('count')
            for k,v in detail.items():components[k]=components.get(k,0.)+v
        accuracy=validation_accuracy()
        row=dict(epoch=epoch+1,lr=optimizer.param_groups[0]['lr'],train_loss=total/n,train_accuracy=correct/count,validation_accuracy=accuracy,**{k:v/batches for k,v in components.items()})
        history.append(row)
        if accuracy>best:
            best=accuracy
            torch.save(dict(model_state=model.state_dict(),epoch=epoch+1,validation_accuracy=accuracy,config=cfg,split=split,classes=train.classes),out/'best.pt')
        scheduler.step();write_json(out/'history.json',history)
        print(row)
    write_json(out/'complete.json',dict(config=cfg,best_validation_accuracy=best,epochs_completed=cfg['epochs']))
    print('Completed',RUN,'selected validation accuracy',best)
train_or_reuse()

Root: d:\LUMS GAMING\6. 2026 Fall\CS 6304\Programming Assignments\PA1\Task 4\task4 Device: cuda
Files already downloaded and verified
Files already downloaded and verified
{'epoch': 1, 'lr': 0.001, 'train_loss': 0.603917400709788, 'train_accuracy': 0.9990666666666667, 'validation_accuracy': 0.9444, 'ce': 0.07965041367854363, 'classifier_placeholder': 0.20601852488471195, 'data_placeholder': 3.1787646565247667, 'pairs': 56.59090909090909}
{'epoch': 2, 'lr': 0.0009990133642141358, 'train_loss': 0.33694170489841035, 'train_accuracy': 0.9994222222222222, 'validation_accuracy': 0.9428, 'ce': 0.10449801465835083, 'classifier_placeholder': 0.02046350069047714, 'data_placeholder': 2.120023297653957, 'pairs': 56.76136363636363}
{'epoch': 3, 'lr': 0.000996057350657239, 'train_loss': 0.32833567004733616, 'train_accuracy': 0.9993777777777778, 'validation_accuracy': 0.947, 'ce': 0.10380354601974515, 'classifier_placeholder': 0.01565232665796595, 'data_placeholder': 2.088926786218177, 'pairs': 56.28